# Explore the LNMC MT Multimodal Single-Cell Dataset (Comprehensive)

## Introduction
This notebook provides a comprehensive exploratory analysis of the **Multimodal Single-Cell Dataset of Juvenile Rat Neocortical Neurons — Morphology, Whole-Cell Patch Electrophysiology, and Source Histological Slices**, produced at the EPFL Laboratory of Neural Microcircuitry (LNMC).

Learn more:
- Data Package DOI: [10.71728/senscience.bt07-sboc](https://sen.science/doi/10.71728/senscience.bt07-sboc/)

As a FAIR² Data Package, it ensures accessibility, interoperability, and AI-readiness. FAIR² datasets follow the MLCommons **Croissant** 🥐 format for machine learning datasets. See the [MLCommons Croissant Format Specification](https://docs.mlcommons.org/croissant/docs/croissant-spec.html).

The dataset pairs three modalities cell-by-cell across 396 curated slices:
- **244 morphological reconstructions** in SWC format (~3.9 M nodes total)
- **641 whole-cell patch-clamp recordings** in NWB (HDF5) plus decimated sweep JSON
- **244 histological slide images** in JPEG

Six tidy Parquet record-sets describe the modalities: cell morphology metadata + per-node coordinates; ephys session metadata + per-sweep acquisitions; and the slice-level slide-image manifest + experimenter's archival log. This notebook will load them all via the Croissant metadata, then render interactive viewers — one for the 3-D morphology (SWC), one for the current-clamp sweep traces, and one for the slide-image inventory — for a representative cell.

### Install and import required libraries

In [ ]:
# Install mlcroissant and the plotting/loading dependencies
!pip install mlcroissant pandas pyarrow matplotlib seaborn plotly matplotlib-venn

In [ ]:
import json
from pathlib import Path

import mlcroissant as mlc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import Markdown, Image, display

sns.set_theme(style="whitegrid")

## 1. Data Loading
We use the `mlcroissant` library to read the FAIR² `fair2.json` metadata, then load the six tidy Parquet record-sets it declares into `pandas` DataFrames. Every column in every table is linked back to a specific method step via the `method_description_object` entries in the data dictionary.

In [ ]:
# Load the dataset from the Croissant metadata file
# (uncomment the URL form if running remotely)
# ds = mlc.Dataset('https://sen.science/doi/10.71728/senscience.bt07-sboc/fair2.json')
ds = mlc.Dataset('./fair2.json')

# Load all record sets into a dictionary of pandas DataFrames
dataframes = {}
for rs in ds.metadata.record_sets:
    print(f"Loading record set: {rs.name}")
    records = ds.records(rs.id)
    df = pd.DataFrame(list(records))
    # mlcroissant produces column names like 'record_set/column'. Strip the prefix.
    df.columns = [col.split('/')[-1] for col in df.columns]
    dataframes[rs.name] = df

# Assign to descriptive variables
cell_meta       = dataframes['cell_morphology_metadata.parquet']
cell_nodes      = dataframes['cell_morphology_nodes.parquet']
ephys_meta      = dataframes['ephys_metadata.parquet']
ephys_acq       = dataframes['ephys_acquisitions.parquet']
slice_manifest  = dataframes['slice_image_manifest.parquet']
slides_log      = dataframes['archival_slides_log.parquet']

# mlcroissant sometimes loads text columns as bytes — normalise here
for df in (cell_meta, cell_nodes, ephys_meta, ephys_acq, slice_manifest, slides_log):
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

print(f"\nAll {len(dataframes)} record sets loaded successfully via mlcroissant.")
print(f"  cell_morphology_metadata : {len(cell_meta):>9,} rows × {len(cell_meta.columns)} cols")
print(f"  cell_morphology_nodes    : {len(cell_nodes):>9,} rows × {len(cell_nodes.columns)} cols")
print(f"  ephys_metadata           : {len(ephys_meta):>9,} rows × {len(ephys_meta.columns)} cols")
print(f"  ephys_acquisitions       : {len(ephys_acq):>9,} rows × {len(ephys_acq.columns)} cols")
print(f"  slice_image_manifest     : {len(slice_manifest):>9,} rows × {len(slice_manifest.columns)} cols")
print(f"  archival_slides_log      : {len(slides_log):>9,} rows × {len(slides_log.columns)} cols")

### 1.1 Data dictionary with method-step provenance
Every column in every table is linked to a specific method step via the schema.json's `method_description_object`. Here we surface the linkage in tabular form.

In [ ]:
# Load the schema.json to expose the data dictionary + method step links
schema = json.loads(Path('./schema.json').read_text())

# Method-step catalogue
step_index = {}
for section in schema.get('methods_description', []):
    for step in section.get('steps', []):
        step_index[step['id']] = {
            'section': section.get('name', ''),
            'step':    step.get('name', ''),
        }
print(f'{len(step_index)} method steps declared in schema.methods_description')

# Build a wide table: (table, column, description, → method section / step)
rows = []
for row in schema['data_dictionary']['data_dictionary_rows']:
    tname = row['file_object']['name'].replace('.parquet', '')
    for f in row['fields']:
        md_obj = f.get('method_description_object', {}) or {}
        rows.append({
            'table':     tname,
            'column':    f['variable_name'],
            'dataType':  f.get('dataType', '').split('/')[-1],
            'unit':      f.get('unit', {}).get('symbol', ''),
            'method_step': md_obj.get('name', '—'),
        })
dd_df = pd.DataFrame(rows)
display(Markdown(f'### {len(dd_df)} variables across {dd_df.table.nunique()} tables — each linked to a method step'))
display(dd_df.head(15))

### 1.2 Dataset Metadata
Inspect the dataset's top-level metadata to confirm name, DOI, and licence.

In [ ]:
metadata = ds.metadata.to_json()

name        = metadata.get('name', 'N/A')
description = metadata.get('description', 'N/A')
if isinstance(description, list):
    description = description[0].get('@value', description[0]) if description else 'N/A'

print(f"Name       : {name}")
print(f"Version    : {metadata.get('version', 'N/A')}")
print(f"License    : {metadata.get('license', 'N/A')}")
print(f"URL        : {metadata.get('url', 'N/A')}")
print(f"Creators   : {len(ds.metadata.creators)}")
for c in ds.metadata.creators:
    print(f"    - {c.name}")
print(f"\nDescription: {description}")

## 2. Data Preparation and Integration
We assemble a per-cell integrated summary by joining the four record-sets on the canonical cell identifiers, and derive per-modality summary statistics (nodes per reconstruction, sweeps per recording).

In [ ]:
# --- 1. Per-cell morphology summary --- #
# cell_morphology_nodes has one row per SWC node; summarise to one row per cell.
morph_nodes_summary = (
    cell_nodes.groupby('cell_name')
              .agg(n_nodes=('node_id', 'size'),
                   n_types=('type', 'nunique'))
              .reset_index()
              .rename(columns={'cell_name': 'name'})
)

# Compartment breakdown per cell (type 1=soma, 2=axon, 3=basal, 4=apical)
comp_labels = {1: 'soma', 2: 'axon', 3: 'basal', 4: 'apical'}
comp_wide = (cell_nodes
             .assign(compartment=cell_nodes['type'].map(comp_labels))
             .groupby(['cell_name', 'compartment']).size()
             .unstack(fill_value=0)
             .reset_index()
             .rename(columns={'cell_name': 'name'}))
morph_summary = cell_meta.merge(morph_nodes_summary, on='name', how='left')
morph_summary = morph_summary.merge(comp_wide, on='name', how='left')

# --- 2. Per-recording sweep summary --- #
ephys_sweep_summary = (
    ephys_acq.groupby('session_id')
             .agg(n_sweeps=('sweep_number', 'size'),
                  n_protocols=('stimulus_type', 'nunique'),
                  mean_rate_hz=('rate_hz', 'mean'))
             .reset_index()
             .rename(columns={'session_id': 'name'})
)
ephys_summary = ephys_meta.merge(ephys_sweep_summary, on='name', how='left')

display(Markdown('### Morphology summary (per cell)'))
display(morph_summary.head())
display(Markdown('### Electrophysiology summary (per recording)'))
display(ephys_summary.head())

## 3. Exploratory Data Analysis (EDA)

### 3.1 Overview of the Integrated Dataset

In [ ]:
print("Morphology summary:")
print(f"  {morph_summary.shape[0]} cells × {morph_summary.shape[1]} fields")
print(f"  species distribution: {dict(morph_summary['species_name'].value_counts())}")
print()
print("Ephys summary:")
print(f"  {ephys_summary.shape[0]} recordings × {ephys_summary.shape[1]} fields")
print(f"  species distribution: {dict(ephys_summary['species_name'].value_counts())}")
print()
n_covered = ephys_acq['session_id'].nunique()
print(f"Sweep-level acquisitions: {len(ephys_acq):,} rows across {n_covered} sessions")
print(f"Node-level rows        : {len(cell_nodes):,} across {cell_nodes['cell_name'].nunique()} cells")

### 3.2 Data Visualization

#### 3.2.1 Morphology composition
Distribution of nodes per reconstruction and compartment breakdown across the curated SWC set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# (a) histogram of nodes-per-cell (log x)
axes[0].hist(morph_summary['n_nodes'].dropna(),
             bins=30, color='#4C72B0', edgecolor='white')
axes[0].set_xscale('log')
axes[0].set_xlabel('Nodes per reconstruction (log scale)')
axes[0].set_ylabel('# cells')
axes[0].set_title('Distribution of SWC nodes per reconstruction')
axes[0].grid(True, which="both", ls="--", alpha=.3)

# (b) compartment composition — stacked share across all cells
comp_totals = morph_summary[['soma', 'axon', 'basal', 'apical']].sum()
comp_totals = comp_totals / comp_totals.sum() * 100
axes[1].bar(comp_totals.index, comp_totals.values,
            color=['#000000', '#E74C3C', '#3498DB', '#F39C12'])
for i, v in enumerate(comp_totals.values):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha='center', fontsize=10)
axes[1].set_ylabel('% of total SWC nodes')
axes[1].set_title('Compartment composition across the curated bundle')

plt.tight_layout(); plt.show()

#### 3.2.2 Morphology 3-D viewer
Interactive point-cloud rendering of a representative reconstruction, coloured by SWC compartment type. We read the SWC file directly from the `swc/` FileSet declared in `fair2.json`.

You can change `EXAMPLE_CELL` below to any cell name that appears in `cell_morphology_metadata`.

In [ ]:
EXAMPLE_CELL = 'C210401D'   # cell with a spike-free IV example — used in Fig 3

# The SWC FileSet is a directory tree. Look up the on-disk path via the record.
row = morph_summary[morph_summary['name'].str.contains(EXAMPLE_CELL, na=False, regex=False)].head(1)
if row.empty:
    # Pick the first cell that has on-disk node coverage as a fallback
    with_nodes = morph_summary[morph_summary['n_nodes'].notna()]
    row = with_nodes.iloc[[0]]
CELL_NAME = row['name'].iloc[0]
SWC_PATH  = Path('.') / row['swc_file'].iloc[0].lstrip('assets/')
print(f'Rendering: {CELL_NAME}  ({SWC_PATH})')

# Parse SWC — 7 whitespace-separated columns:
#   node_id  type  x  y  z  radius  parent_id
try:
    swc = pd.read_csv(SWC_PATH, sep=r'\s+', comment='#',
                      header=None,
                      names=['node_id', 'type', 'x', 'y', 'z', 'radius', 'parent_id'])
    src = 'file'
except (FileNotFoundError, OSError):
    # Fallback to the nodes table
    swc = cell_nodes[cell_nodes['cell_name'] == CELL_NAME].copy()
    src = 'parquet'

print(f'  {len(swc):,} nodes loaded from {src}')

# 3-D interactive plot with plotly
type_labels = {1: 'soma', 2: 'axon', 3: 'basal dendrite', 4: 'apical dendrite'}
type_colors = {1: '#000000', 2: '#E74C3C', 3: '#3498DB', 4: '#F39C12'}
swc['compartment'] = swc['type'].map(type_labels).fillna('other')
swc['color']       = swc['type'].map(type_colors).fillna('#7F8C8D')

fig = go.Figure()
for t, sub in swc.groupby('type'):
    fig.add_trace(go.Scatter3d(
        x=sub['x'], y=sub['y'], z=sub['z'],
        mode='markers',
        marker=dict(size=1.4, color=type_colors.get(t, '#7F8C8D')),
        name=type_labels.get(t, f'type {t}'),
        hovertext=[f"node {n}" for n in sub['node_id']]
    ))
fig.update_layout(
    title=f'Morphology · {CELL_NAME}  (n={len(swc):,} SWC nodes)',
    scene=dict(aspectmode='data',
               xaxis_title='x (µm)', yaxis_title='y (µm)', zaxis_title='z (µm)'),
    height=650, margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(itemsizing='constant')
)
fig.show()

#### 3.2.3 Electrophysiology sweep viewer
Multi-protocol view of the current-clamp responses for the same cell. Each panel shows one stimulus protocol; sweeps are colour-graded within a protocol from earliest (dark) to latest (light).

Data are read from the decimated sweep JSON in the `sweeps/` FileSet — the same input the multi-modal browser uses.

In [ ]:
# Find sweep JSON files under sweeps/ for this cell
# The sweep artifact path is under sweeps/<slice_id>/<stem>.json where <stem>
# matches ephys_metadata.name (e.g. 'C210401D-MT-C1').
sweep_recs = ephys_summary[ephys_summary['name'].str.contains(EXAMPLE_CELL, na=False, regex=False)]
if sweep_recs.empty:
    sweep_recs = ephys_summary.head(1)
SWEEP_NAME = sweep_recs['name'].iloc[0]

slice_id = sweep_recs['cell_base_name'].iloc[0]
sweep_path = Path('.') / 'resources' / 'artifacts' / 'sweeps' / slice_id / f'{SWEEP_NAME}.json'
if not sweep_path.exists():
    # Try to find by scanning
    candidates = list(Path('.').rglob(f'{SWEEP_NAME}.json'))
    if candidates:
        sweep_path = candidates[0]
print(f'Loading sweeps: {sweep_path}')

with open(sweep_path) as f:
    sweep_doc = json.load(f)

protocols = sweep_doc.get('protocols', [])
sweeps    = sweep_doc.get('sweeps', [])
print(f'  {len(sweeps)} sweeps across {len(protocols)} protocols')

# Show top-8 protocols by sweep count (adaptive)
by_protocol = {}
for s in sweeps:
    by_protocol.setdefault(s.get('protocol', 'unknown'), []).append(s)
top_protocols = sorted(by_protocol.items(), key=lambda kv: -len(kv[1]))[:8]

n_panels = len(top_protocols)
n_cols = 2
n_rows = (n_panels + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.3 * n_rows), squeeze=False)
axes = axes.ravel()

cmap = plt.get_cmap('viridis')
for ax_idx, (proto, plist) in enumerate(top_protocols):
    ax = axes[ax_idx]
    n = len(plist)
    for i, sw in enumerate(plist):
        pts = np.asarray(sw['points'])
        if pts.size == 0:
            continue
        ax.plot(pts[:, 0], pts[:, 1], lw=0.6, color=cmap(i / max(n - 1, 1)))
    ax.set_title(f'{proto} · {n} sweep{"s" if n != 1 else ""}')
    ax.set_xlabel('time (s)')
    ax.set_ylabel('V (mV)')
    ax.grid(True, ls='--', alpha=.3)

for ax in axes[len(top_protocols):]:
    ax.axis('off')

fig.suptitle(f'Ephys sweeps · {SWEEP_NAME}  ({len(sweeps)} total)', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

#### 3.2.4 Slide-archive inventory
The two slice-level tables — `slice_image_manifest` and `archival_slides_log` — describe every histological slide acquired and archived by the LNMC experimenters. Below we summarise the slide inventory by magnification and annotation status, and inspect the archival log by experimenter.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# (a) magnification distribution
mag_counts = slice_manifest['magnification'].value_counts()
axes[0].bar(mag_counts.index.astype(str), mag_counts.values, color='#4C72B0')
for i, v in enumerate(mag_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center')
axes[0].set_title('Slide images by magnification')
axes[0].set_ylabel('# slides')

# (b) annotated vs raw
ann_counts = slice_manifest['is_annotated'].value_counts().sort_index()
labels = ['raw' if x is False or x == 0 else 'annotated' for x in ann_counts.index]
axes[1].pie(ann_counts.values, labels=[f'{l} (n={v})' for l, v in zip(labels, ann_counts.values)],
             autopct='%1.1f%%', colors=['#DFE6EC', '#F39C12'])
axes[1].set_title('Annotated vs raw slides')

# (c) archival log by experimenter
exp_counts = slides_log['experimenter'].value_counts().head(10)
axes[2].barh(exp_counts.index[::-1], exp_counts.values[::-1], color='#3498DB')
axes[2].set_title('Archival-log rows per experimenter (top 10)')
axes[2].set_xlabel('# archived slides')

plt.tight_layout(); plt.show()

display(Markdown('#### Slide-inventory summary'))
display(slice_manifest.head())
display(Markdown('#### Archival-log summary (first 5 rows)'))
display(slides_log.head())

#### 3.2.5 Cross-modal coverage
Distribution of the 396 curated slices across morphology (M), electrophysiology (E), and slide-image (S) modalities — a Venn diagram of which cells carry which measurements.

In [ ]:
from matplotlib_venn import venn3

# Build the modality-set per slice
morph_slices = set(morph_summary['cell_base_name'].dropna().astype(str))
ephys_slices = set(ephys_summary['cell_base_name'].dropna().astype(str))

# Slide-image slices come from the records.json in the artifacts dir
try:
    recs = json.load(open('./resources/artifacts/records.json'))
    slide_slices = {r['cell_id'] for r in recs if r.get('slide_image')}
except FileNotFoundError:
    slide_slices = set()

fig, ax = plt.subplots(figsize=(9, 7))
venn3([morph_slices, ephys_slices, slide_slices],
      set_labels=('Morphology (SWC)', 'Electrophysiology (NWB)', 'Slide image'),
      ax=ax)
ax.set_title(f'Cross-modal coverage — {len(morph_slices | ephys_slices | slide_slices)} unique slices')
plt.show()

print(f'M only          : {len(morph_slices - ephys_slices - slide_slices)}')
print(f'E only          : {len(ephys_slices - morph_slices - slide_slices)}')
print(f'S only          : {len(slide_slices - morph_slices - ephys_slices)}')
print(f'M ∩ E           : {len(morph_slices & ephys_slices - slide_slices)}')
print(f'M ∩ S           : {len(morph_slices & slide_slices - ephys_slices)}')
print(f'E ∩ S           : {len(ephys_slices & slide_slices - morph_slices)}')
print(f'All three (M∩E∩S): {len(morph_slices & ephys_slices & slide_slices)}')

## 4. Conclusion
This notebook walked through a first exploration of the LNMC MT multimodal single-cell dataset:

- **Loading**: Croissant metadata (`fair2.json`) drove the ingestion of six tidy Parquet record-sets — 244 morphology metadata rows, 641 electrophysiology metadata rows, their respective per-node and per-sweep expansion tables, and the two slice-level tables (348-row image manifest + 420-row archival log).
- **Data dictionary with method provenance**: every column in every table is linked to a specific step in `methods_description` via `method_description_object`, matching the atolls exemplar pattern.
- **Morphology viewer**: 3-D interactive rendering of a single reconstruction, coloured by SWC compartment (soma, axon, basal dendrite, apical dendrite), using either the on-disk `.swc` file or the tidy nodes Parquet as a fallback source.
- **Electrophysiology viewer**: multi-panel current-clamp trace display grouped by stimulus protocol, driven by the decimated sweep JSON in the `sweeps/` FileSet — the same input the multi-modal browser uses.
- **Slide-archive inventory**: magnification distribution, annotated-vs-raw split, and per-experimenter archival-log breakdown.
- **Cross-modal coverage**: Venn diagram summarising how the 396 curated slices distribute across morphology, ephys, and slide-image modalities.

Everything shown here is reproducible from the FAIR² package alone: the Croissant `recordSet` entries define the tabular schema (with method-step linkage), and the `fileSet` entries define the browseable per-slice-dir asset trees for SWC, sweeps, and slide imagery. To extend, consider (a) automated e-type / m-type classification driven by the sweep viewer's protocol grouping, (b) biophysically-detailed single-neuron modelling fit to the standardised stimulus battery, or (c) machine-learning studies of the structure–function relationship across the paired cells.